In [ ]:
from pathlib import Path
import json
import pandas as pd

from src.data.load import read_text
from src.data.preprocess import normalize_text
from src.graphs.extract_events import extract_events, events_to_dict
from src.graphs.build_graph import build_event_graph, graph_to_jsonable

# Paths
meta_path = Path("data/metadata.csv")
data_root = Path("data/human")
out_root = Path("outputs/human_graphs")
out_root.mkdir(parents=True, exist_ok=True)

# Load metadata
meta = pd.read_csv(meta_path)

# Minimal required columns check
required_cols = {"story_id", "culture"}
missing = required_cols - set(meta.columns)
if missing:
    raise ValueError(f"metadata.csv is missing columns: {missing}. Required: {required_cols}")

rows = []
errors = []

for _, r in meta.iterrows():
    story_id = str(r["story_id"])
    culture = str(r["culture"])

    story_path = data_root / culture / f"{story_id}.txt"
    if not story_path.exists():
        errors.append({"story_id": story_id, "culture": culture, "error": f"Missing file: {story_path}"})
        continue

    try:
        raw = read_text(story_path)
        text = normalize_text(raw)

        events = extract_events(text)
        G = build_event_graph(events)

        story_out = out_root / story_id
        story_out.mkdir(parents=True, exist_ok=True)

        # Save events and graph
        (story_out / "events.json").write_text(json.dumps(events_to_dict(events), indent=2), encoding="utf-8")
        (story_out / "graph.json").write_text(json.dumps(graph_to_jsonable(G), indent=2), encoding="utf-8")

        # Basic stats (useful for your paper + debugging)
        word_count = len(text.split())
        rows.append({
            "story_id": story_id,
            "culture": culture,
            "word_count": word_count,
            "num_events": len(events),
            "events_per_100_words": (len(events) / word_count * 100) if word_count else 0.0,
            "input_path": str(story_path),
            "output_dir": str(story_out),
        })

    except Exception as e:
        errors.append({"story_id": story_id, "culture": culture, "error": repr(e)})

# Save summary tables
summary_df = pd.DataFrame(rows).sort_values(["culture", "story_id"])
summary_df.to_csv(out_root / "summary.csv", index=False)

errors_df = pd.DataFrame(errors)
errors_df.to_csv(out_root / "errors.csv", index=False)

print(f"Done. Processed {len(summary_df)} stories.")
print(f"Saved: {out_root / 'summary.csv'}")
if len(errors_df) > 0:
    print(f"Some stories failed. See: {out_root / 'errors.csv'}")
    display(errors_df.head(10))
else:
    print("No errors 🎉")

display(summary_df.head(10))

Done. Processed 30 stories.
Saved: outputs\human_graphs\summary.csv
No errors 🎉


,story_id,culture,word_count,num_events,events_per_100_words,input_path,output_dir
0,g01,german,2530,76,3.003953,data\human\german\g01.txt,outputs\human_graphs\g01
1,g02,german,2932,132,4.502046,data\human\german\g02.txt,outputs\human_graphs\g02
2,g03,german,2341,100,4.271679,data\human\german\g03.txt,outputs\human_graphs\g03
3,g04,german,1194,38,3.182580,data\human\german\g04.txt,outputs\human_graphs\g04
4,g05,german,1139,43,3.775241,data\human\german\g05.txt,outputs\human_graphs\g05
5,g06,german,1605,52,3.239875,data\human\german\g06.txt,outputs\human_graphs\g06
6,g07,german,2120,117,5.518868,data\human\german\g07.txt,outputs\human_graphs\g07
7,g08,german,1585,60,3.785489,data\human\german\g08.txt,outputs\human_graphs\g08
8,g09,german,2340,75,3.205128,data\human\german\g09.txt,outputs\human_graphs\g09
9,g10,german,2414,95,3.935377,data\human\german\g10.txt,outputs\human_graphs\g10
